In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# -------------------------------
# LOAD DATA
# -------------------------------
df = pd.read_csv(
    r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_modified_with_anomalies.csv"
)
print('Initial Shape:', df.shape)

# -------------------------------
# 1. HANDLE DUPLICATES
# -------------------------------
df.drop_duplicates(inplace=True)
print('Shape after dropping duplicates:', df.shape)

# -------------------------------
# 2. HANDLE MISSING VALUES
# -------------------------------
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# -------------------------------
# 3. OUTLIER TREATMENT
# -------------------------------
for col in ['Market_Price', 'Purchase_Amount', 'Competitor_Rating', 'Competitor_Price']:
    if col in df.columns:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = np.clip(df[col], lower, upper)

# -------------------------------
# 4. FEATURE ENGINEERING
# -------------------------------
if 'Purchase_Date' in df.columns:
    df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'], errors='coerce')
    df['Year'] = df['Purchase_Date'].dt.year
    df['Month'] = df['Purchase_Date'].dt.month
    df['Day'] = df['Purchase_Date'].dt.day
    df['WeekOfYear'] = df['Purchase_Date'].dt.isocalendar().week
    df['DayOfWeek'] = df['Purchase_Date'].dt.dayofweek
    df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Fix Promotion_Competitor
if 'Promotion_Competitor' in df.columns:
    df['Promotion_Competitor'] = pd.to_numeric(df['Promotion_Competitor'], errors='coerce').fillna(0)
    df['Promotion_Flag'] = (df['Promotion_Competitor'] > 0).astype(int)

# -------------------------------
# 5. CREATE AGENT 2 TARGET
# -------------------------------
if 'Current_Stock' not in df.columns:
    np.random.seed(42)
    df['Current_Stock'] = np.random.randint(50, 200, size=len(df))

df['Forecasted_Purchase_Amount'] = df['Purchase_Amount']
df['Stock_Risk'] = (df['Forecasted_Purchase_Amount'] > df['Current_Stock']).astype(int)

# -------------------------------
# 6. ENCODING CATEGORICAL VARIABLES (MEMORY EFFICIENT)
# -------------------------------
# Only keep columns that exist in df
cat_cols_existing = [col for col in cat_cols if col in df.columns]

# One-hot for low-cardinality (<20 unique)
low_card_cols = [col for col in cat_cols_existing if df[col].nunique() < 20]
df = pd.get_dummies(df, columns=low_card_cols, drop_first=True)

# Frequency encoding for high-cardinality (>20 unique)
high_card_cols = [col for col in cat_cols_existing if col not in low_card_cols and col in df.columns]
for col in high_card_cols:
    freq_map = df[col].value_counts(normalize=True)
    df[col + '_freq'] = df[col].map(freq_map)
    df.drop(col, axis=1, inplace=True)

# -------------------------------
# 7. SCALE NUMERIC FEATURES
# -------------------------------
scaler = StandardScaler()
scaled_cols = ['Market_Price', 'Purchase_Amount', 'Competitor_Rating', 'Competitor_Price', 
               'Current_Stock', 'Forecasted_Purchase_Amount']
for col in scaled_cols:
    if col in df.columns:
        df[col] = scaler.fit_transform(df[[col]])

# -------------------------------
# 8. SAVE CSV FILES
# -------------------------------
# Full preprocessed
df.to_csv(r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_preprocessed_full.csv", index=False)

# Agent 1
agent1_features = [c for c in df.columns if c not in ['Purchase_Amount', 'Stock_Risk', 'Forecasted_Purchase_Amount']]
df_agent1 = df[agent1_features + ['Purchase_Amount']]
df_agent1.to_csv(r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_agent1.csv", index=False)

# Agent 2
agent2_features = [c for c in df.columns if c not in ['Purchase_Amount', 'Stock_Risk']]
df_agent2 = df[agent2_features + ['Stock_Risk']]
df_agent2.to_csv(r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_agent2.csv", index=False)

print("Preprocessing complete. CSV files saved!")


Initial Shape: (52500, 20)
Shape after dropping duplicates: (50187, 20)


C:\Users\USER\AppData\Local\Temp\ipykernel_12540\3753652130.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


Preprocessing complete. CSV files saved!
